# 🗄️ SQL Complete Syntax & Querying Guide

**A bite-sized, practical reference covering standard ANSI SQL from DDL/DML to window functions and CTEs.**
*Uses Python's built-in in-memory SQLite engine (no setup required).* 

## 📑 Index (Table of Contents)

1. [1. Database Setup & Query Runner](#1-database-setup--query-runner)
2. [2. Data Definition Language (DDL)](#2-data-definition-language-ddl)
3. [3. Data Manipulation Language (DML) & Upsert](#3-data-manipulation-language-dml--upsert)
4. [4. Data Query Language (DQL) - Basic Select & Filtering](#4-data-query-language-dql---basic-select--filtering)
5. [5. Operators, String, Math & Date Functions](#5-operators-string-math--date-functions)
6. [6. Aggregate Functions & Grouping](#6-aggregate-functions--grouping)
7. [7. Joins & Set Operations](#7-joins--set-operations)
8. [8. Subqueries & Derived Tables](#8-subqueries--derived-tables)
9. [9. Common Table Expressions (CTEs)](#9-common-table-expressions-ctes)
10. [10. Window / Analytic Functions](#10-window--analytic-functions)
11. [11. Views, Triggers & Transactions (TCL)](#11-views-triggers--transactions-tcl)
12. [12. Query Planning & Optimization](#12-query-planning--optimization)

13. [13. Analytical Warehouse Patterns & DuckDB](#13-analytical-warehouse-patterns--duckdb)

---


## 1. Database Setup & Query Runner
### 1.1 In-Memory SQLite Connection & Runner
Creates an in-memory database and helper functions to run and format queries.


In [1]:
import sqlite3

conn = sqlite3.connect(":memory:")
conn.execute("PRAGMA foreign_keys = ON;")

def run_query(sql: str, title: str = None):
    if title:
        print(f"=== {title} ===")
    cur = conn.cursor()
    cur.execute(sql)
    rows = cur.fetchall()
    cols = [d[0] for d in cur.description] if cur.description else []
    if not cols:
        print("(Query executed successfully, no rows returned)\n")
        return
    col_w = [max(len(str(c)), max((len(str(r[i])) for r in rows), default=0)) for i, c in enumerate(cols)]
    print(" | ".join(str(c).ljust(col_w[i]) for i, c in enumerate(cols)))
    print("-+-".join("-" * col_w[i] for i in range(len(cols))))
    for r in rows:
        print(" | ".join(str(r[i]).ljust(col_w[i]) for i in range(len(cols))))
    print(f"({len(rows)} row{'s' if len(rows) != 1 else ''})\n")

def run_exec(sql: str):
    conn.executescript(sql)
    conn.commit()

print("Relational engine initialized and ready.")

Relational engine initialized and ready.


## 2. Data Definition Language (DDL)
### 2.1 `CREATE TABLE` with Constraints
Creating tables with `PRIMARY KEY`, `NOT NULL`, `UNIQUE`, and `CHECK` constraints.


In [2]:
run_exec("""
CREATE TABLE departments (
    dept_id INTEGER PRIMARY KEY,
    dept_name TEXT NOT NULL UNIQUE,
    budget REAL CHECK (budget >= 0)
);
""")
print("Table 'departments' created successfully.")

Table 'departments' created successfully.


### 2.2 `CREATE TABLE` with Foreign Keys & Generated Columns
Foreign key references, default values, and computed columns (`GENERATED ALWAYS AS`).


In [3]:
run_exec("""
CREATE TABLE employees (
    emp_id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT NOT NULL,
    dept_id INTEGER REFERENCES departments(dept_id),
    salary REAL CHECK (salary > 0),
    hire_date TEXT DEFAULT (CURRENT_DATE),
    email TEXT UNIQUE,
    salary_annual REAL GENERATED ALWAYS AS (salary * 12) STORED
);
""")
print("Table 'employees' with generated column created.")

Table 'employees' with generated column created.


### 2.3 `ALTER TABLE`
Adding columns and modifying table structures.


In [4]:
run_exec("""
ALTER TABLE employees ADD COLUMN status TEXT DEFAULT 'Active';
""")
print("Column 'status' added via ALTER TABLE.")

Column 'status' added via ALTER TABLE.


### 2.4 `CREATE INDEX`
Creating non-unique and unique indexes to accelerate query searches.


In [5]:
run_exec("""
CREATE INDEX idx_emp_dept ON employees(dept_id);
CREATE UNIQUE INDEX idx_emp_email ON employees(email);
""")
print("Indexes created.")

Indexes created.


## 3. Data Manipulation Language (DML) & Upsert
### 3.1 `INSERT INTO` (Multiple Rows)
Populating departments and employees with sample data.


In [6]:
run_exec("""
INSERT INTO departments (dept_id, dept_name, budget) VALUES
    (1, 'Engineering', 500000),
    (2, 'Data Platform', 400000),
    (3, 'Marketing', 200000),
    (4, 'Executive', 150000);

INSERT INTO employees (name, dept_id, salary, hire_date, email) VALUES
    ('Alice Smith', 1, 9500, '2021-03-15', 'alice@tech.com'),
    ('Bob Jones', 1, 8500, '2022-06-01', 'bob@tech.com'),
    ('Charlie Brown', 2, 10500, '2020-01-10', 'charlie@tech.com'),
    ('Diana Prince', 2, 11000, '2019-11-20', 'diana@tech.com'),
    ('Evan Wright', 3, 6000, '2023-02-01', 'evan@tech.com'),
    ('Fiona Gallagher', 3, 6500, '2022-09-15', 'fiona@tech.com'),
    ('Temp Contractor', 3, 4000, '2024-01-01', 'temp@tech.com');
""")
print("Sample rows inserted.")

Sample rows inserted.


### 3.2 `UPDATE` Statement
Modifying existing rows with a conditional `WHERE` filter.


In [7]:
run_exec("""
UPDATE employees 
SET salary = salary * 1.05 
WHERE dept_id = 2; -- 5% raise for Data Platform
""")
print("Employees updated.")

Employees updated.


### 3.3 `DELETE` Statement
Removing specific rows matching a condition.


In [8]:
run_exec("""
DELETE FROM employees 
WHERE email = 'temp@tech.com';
""")
print("Temporary contractor deleted.")

Temporary contractor deleted.


### 3.4 Upsert (`INSERT ... ON CONFLICT`)
Inserting a row or updating on primary key/unique collision.


In [9]:
run_exec("""
INSERT INTO departments (dept_id, dept_name, budget)
VALUES (1, 'Engineering', 550000)
ON CONFLICT(dept_id) DO UPDATE SET budget = excluded.budget;
""")

run_query("SELECT dept_id, dept_name, budget FROM departments WHERE dept_id = 1;", "Upserted Department")

=== Upserted Department ===
dept_id | dept_name   | budget  
--------+-------------+---------
1       | Engineering | 550000.0
(1 row)



## 4. Data Query Language (DQL) - Basic Select & Filtering
### 4.1 Basic `SELECT`, Aliases & `DISTINCT`
Projecting specific columns, renaming with `AS`, and finding unique values.


In [10]:
run_query("""
SELECT DISTINCT 
    dept_id AS department_code,
    status
FROM employees;
""", "Distinct Departments & Status")

=== Distinct Departments & Status ===
department_code | status
----------------+-------
1               | Active
2               | Active
3               | Active
(3 rows)



### 4.2 Filtering with `WHERE`, `BETWEEN` & `IN`
Filtering records by ranges and set membership.


In [11]:
run_query("""
SELECT emp_id, name, salary, hire_date
FROM employees
WHERE salary BETWEEN 7000 AND 12000
  AND dept_id IN (1, 2);
""", "High Earners in Dept 1 & 2")

=== High Earners in Dept 1 & 2 ===
emp_id | name          | salary  | hire_date 
-------+---------------+---------+-----------
1      | Alice Smith   | 9500.0  | 2021-03-15
2      | Bob Jones     | 8500.0  | 2022-06-01
3      | Charlie Brown | 11025.0 | 2020-01-10
4      | Diana Prince  | 11550.0 | 2019-11-20
(4 rows)



### 4.3 Pattern Matching (`LIKE` & `%`)
Filtering strings by patterns using wildcard `%` (any characters) and `_` (single character).


In [12]:
run_query("""
SELECT name, email
FROM employees
WHERE name LIKE 'A%' OR email LIKE '%tech.com';
""", "Pattern Match (Starts with A or ends with tech.com)")

=== Pattern Match (Starts with A or ends with tech.com) ===
name            | email           
----------------+-----------------
Alice Smith     | alice@tech.com  
Bob Jones       | bob@tech.com    
Charlie Brown   | charlie@tech.com
Diana Prince    | diana@tech.com  
Evan Wright     | evan@tech.com   
Fiona Gallagher | fiona@tech.com  
(6 rows)



### 4.4 Handling Nulls (`IS NULL`, `COALESCE`)
Testing for NULL and substituting fallback values.


In [13]:
run_query("""
SELECT 
    name,
    COALESCE(email, 'no-email@company.com') AS clean_email
FROM employees
WHERE email IS NOT NULL;
""", "Safe Null Handling with COALESCE")

=== Safe Null Handling with COALESCE ===
name            | clean_email     
----------------+-----------------
Alice Smith     | alice@tech.com  
Bob Jones       | bob@tech.com    
Charlie Brown   | charlie@tech.com
Diana Prince    | diana@tech.com  
Evan Wright     | evan@tech.com   
Fiona Gallagher | fiona@tech.com  
(6 rows)



### 4.5 Sorting & Pagination (`ORDER BY`, `LIMIT`, `OFFSET`)
Ordering results ascending/descending and paginating records.


In [14]:
run_query("""
SELECT name, salary, hire_date
FROM employees
ORDER BY salary DESC, hire_date ASC
LIMIT 3 OFFSET 0;
""", "Top 3 Highest Paid Employees")

=== Top 3 Highest Paid Employees ===
name          | salary  | hire_date 
--------------+---------+-----------
Diana Prince  | 11550.0 | 2019-11-20
Charlie Brown | 11025.0 | 2020-01-10
Alice Smith   | 9500.0  | 2021-03-15
(3 rows)



## 5. Operators, String, Math & Date Functions
### 5.1 String Functions & Concatenation (`||`)
Manipulating text columns with `UPPER`, `SUBSTR`, `LENGTH`, and `||`.


In [15]:
run_query("""
SELECT 
    name,
    UPPER(name) AS name_upper,
    LENGTH(name) AS char_count,
    name || ' <' || email || '>' AS formatted_contact
FROM employees
LIMIT 3;
""", "String Transformations")

=== String Transformations ===
name          | name_upper    | char_count | formatted_contact               
--------------+---------------+------------+---------------------------------
Alice Smith   | ALICE SMITH   | 11         | Alice Smith <alice@tech.com>    
Bob Jones     | BOB JONES     | 9          | Bob Jones <bob@tech.com>        
Charlie Brown | CHARLIE BROWN | 13         | Charlie Brown <charlie@tech.com>
(3 rows)



### 5.2 Math Functions & Expressions
Performing arithmetic and rounding numbers.


In [16]:
run_query("""
SELECT 
    name,
    salary,
    ROUND(salary / 1000.0, 1) AS salary_k,
    ROUND(salary * 1.10, 2) AS with_10pct_bonus
FROM employees
LIMIT 3;
""", "Math & Rounding Functions")

=== Math & Rounding Functions ===
name          | salary  | salary_k | with_10pct_bonus
--------------+---------+----------+-----------------
Alice Smith   | 9500.0  | 9.5      | 10450.0         
Bob Jones     | 8500.0  | 8.5      | 9350.0          
Charlie Brown | 11025.0 | 11.0     | 12127.5         
(3 rows)



### 5.3 Date & Time Functions
Calculating current dates, extracting years with `strftime`, and calculating tenure.


In [17]:
run_query("""
SELECT 
    name,
    hire_date,
    strftime('%Y', hire_date) AS hire_year,
    CAST(strftime('%Y', 'now') AS INT) - CAST(strftime('%Y', hire_date) AS INT) AS tenure_years
FROM employees
LIMIT 3;
""", "Date Calculations")

=== Date Calculations ===
name          | hire_date  | hire_year | tenure_years
--------------+------------+-----------+-------------
Alice Smith   | 2021-03-15 | 2021      | 5           
Bob Jones     | 2022-06-01 | 2022      | 4           
Charlie Brown | 2020-01-10 | 2020      | 6           
(3 rows)



### 5.4 Conditional Expressions (`CASE WHEN`)
Branching logic inside SQL statements.


In [18]:
run_query("""
SELECT 
    name,
    salary,
    CASE 
        WHEN salary >= 11000 THEN 'Tier 1 (Senior/Staff)'
        WHEN salary >= 8500  THEN 'Tier 2 (Mid-Level)'
        ELSE 'Tier 3 (Associate)'
    END AS compensation_tier
FROM employees;
""", "CASE Expression Tiering")

=== CASE Expression Tiering ===
name            | salary  | compensation_tier    
----------------+---------+----------------------
Alice Smith     | 9500.0  | Tier 2 (Mid-Level)   
Bob Jones       | 8500.0  | Tier 2 (Mid-Level)   
Charlie Brown   | 11025.0 | Tier 1 (Senior/Staff)
Diana Prince    | 11550.0 | Tier 1 (Senior/Staff)
Evan Wright     | 6000.0  | Tier 3 (Associate)   
Fiona Gallagher | 6500.0  | Tier 3 (Associate)   
(6 rows)



## 6. Aggregate Functions & Grouping
### 6.1 Basic Aggregates
Calculating `COUNT`, `SUM`, `AVG`, `MIN`, and `MAX` across the entire table.


In [19]:
run_query("""
SELECT 
    COUNT(*) AS total_employees,
    MIN(salary) AS min_salary,
    MAX(salary) AS max_salary,
    ROUND(AVG(salary), 2) AS avg_salary,
    ROUND(SUM(salary), 2) AS total_payroll
FROM employees;
""", "Company-wide Salary Aggregations")

=== Company-wide Salary Aggregations ===
total_employees | min_salary | max_salary | avg_salary | total_payroll
----------------+------------+------------+------------+--------------
6               | 6000.0     | 11550.0    | 8845.83    | 53075.0      
(1 row)



### 6.2 `GROUP BY` Clause
Grouping rows by department to compute group-level metrics.


In [20]:
run_query("""
SELECT 
    dept_id,
    COUNT(*) AS employee_count,
    ROUND(AVG(salary), 2) AS avg_dept_salary,
    ROUND(SUM(salary), 2) AS dept_payroll
FROM employees
GROUP BY dept_id;
""", "Metrics Grouped by Department")

=== Metrics Grouped by Department ===
dept_id | employee_count | avg_dept_salary | dept_payroll
--------+----------------+-----------------+-------------
1       | 2              | 9000.0          | 18000.0     
2       | 2              | 11287.5         | 22575.0     
3       | 2              | 6250.0          | 12500.0     
(3 rows)



### 6.3 `HAVING` vs `WHERE` Clause
`WHERE` filters individual rows before grouping; `HAVING` filters aggregated groups.


In [21]:
run_query("""
SELECT 
    dept_id,
    COUNT(*) AS employee_count,
    ROUND(AVG(salary), 2) AS avg_salary
FROM employees
WHERE status = 'Active'                  -- Filter rows before grouping
GROUP BY dept_id
HAVING COUNT(*) >= 2 AND AVG(salary) > 7000;  -- Filter groups after aggregation
""", "HAVING Filter on Grouped Results")

=== HAVING Filter on Grouped Results ===
dept_id | employee_count | avg_salary
--------+----------------+-----------
1       | 2              | 9000.0    
2       | 2              | 11287.5   
(2 rows)



## 7. Joins & Set Operations
### 7.1 `INNER JOIN`
Combining rows from two tables where keys match.


In [22]:
run_query("""
SELECT 
    e.name AS employee_name,
    d.dept_name,
    e.salary
FROM employees e
INNER JOIN departments d ON e.dept_id = d.dept_id;
""", "INNER JOIN: Employees with Department Names")

=== INNER JOIN: Employees with Department Names ===
employee_name   | dept_name     | salary 
----------------+---------------+--------
Alice Smith     | Engineering   | 9500.0 
Bob Jones       | Engineering   | 8500.0 
Charlie Brown   | Data Platform | 11025.0
Diana Prince    | Data Platform | 11550.0
Evan Wright     | Marketing     | 6000.0 
Fiona Gallagher | Marketing     | 6500.0 
(6 rows)



### 7.2 `LEFT OUTER JOIN`
Preserving all rows from the left table even if no match exists in the right table.


In [23]:
run_query("""
SELECT 
    d.dept_name,
    d.budget,
    e.name AS employee_name
FROM departments d
LEFT JOIN employees e ON d.dept_id = e.dept_id
ORDER BY d.dept_name;
""", "LEFT JOIN: Showing Departments With & Without Staff")

=== LEFT JOIN: Showing Departments With & Without Staff ===
dept_name     | budget   | employee_name  
--------------+----------+----------------
Data Platform | 400000.0 | Charlie Brown  
Data Platform | 400000.0 | Diana Prince   
Engineering   | 550000.0 | Alice Smith    
Engineering   | 550000.0 | Bob Jones      
Executive     | 150000.0 | None           
Marketing     | 200000.0 | Evan Wright    
Marketing     | 200000.0 | Fiona Gallagher
(7 rows)



### 7.3 `CROSS JOIN`
Cartesian product combining every row from table A with every row from table B.


In [24]:
run_query("""
SELECT d.dept_name, q.quarter
FROM departments d
CROSS JOIN (SELECT 'Q1' AS quarter UNION SELECT 'Q2') q
LIMIT 6;
""", "CROSS JOIN: Department Planning Matrix")

=== CROSS JOIN: Department Planning Matrix ===
dept_name     | quarter
--------------+--------
Data Platform | Q1     
Data Platform | Q2     
Engineering   | Q1     
Engineering   | Q2     
Executive     | Q1     
Executive     | Q2     
(6 rows)



### 7.4 Set Operations (`UNION`, `INTERSECT`, `EXCEPT`)
Combining, intersecting, and subtracting result sets.


In [25]:
run_query("""
-- Dept IDs that exist in departments EXCEPT those present in employees
SELECT dept_id FROM departments
EXCEPT
SELECT DISTINCT dept_id FROM employees;
""", "EXCEPT: Departments with 0 employees")

=== EXCEPT: Departments with 0 employees ===
dept_id
-------
4      
(1 row)



## 8. Subqueries & Derived Tables
### 8.1 Scalar Subquery in `SELECT`
Evaluating a single-value subquery in the projection.


In [26]:
run_query("""
SELECT 
    name,
    salary,
    ROUND((SELECT AVG(salary) FROM employees), 2) AS company_avg_salary,
    ROUND(salary - (SELECT AVG(salary) FROM employees), 2) AS diff_from_avg
FROM employees;
""", "Scalar Subquery: Salary Difference from Global Mean")

=== Scalar Subquery: Salary Difference from Global Mean ===
name            | salary  | company_avg_salary | diff_from_avg
----------------+---------+--------------------+--------------
Alice Smith     | 9500.0  | 8845.83            | 654.17       
Bob Jones       | 8500.0  | 8845.83            | -345.83      
Charlie Brown   | 11025.0 | 8845.83            | 2179.17      
Diana Prince    | 11550.0 | 8845.83            | 2704.17      
Evan Wright     | 6000.0  | 8845.83            | -2845.83     
Fiona Gallagher | 6500.0  | 8845.83            | -2345.83     
(6 rows)



### 8.2 Subquery in `WHERE` with `IN`
Filtering using a subquery result list.


In [27]:
run_query("""
SELECT name, dept_id, salary
FROM employees
WHERE dept_id IN (
    SELECT dept_id FROM departments WHERE budget >= 400000
);
""", "Employees in High-Budget Departments")

=== Employees in High-Budget Departments ===
name          | dept_id | salary 
--------------+---------+--------
Alice Smith   | 1       | 9500.0 
Bob Jones     | 1       | 8500.0 
Charlie Brown | 2       | 11025.0
Diana Prince  | 2       | 11550.0
(4 rows)



### 8.3 Correlated Subquery with `EXISTS`
Subquery referencing values from the outer query.


In [28]:
run_query("""
SELECT d.dept_id, d.dept_name
FROM departments d
WHERE EXISTS (
    SELECT 1 FROM employees e 
    WHERE e.dept_id = d.dept_id AND e.salary >= 10000
);
""", "EXISTS: Departments with Staff Earning >= $10k")

=== EXISTS: Departments with Staff Earning >= $10k ===
dept_id | dept_name    
--------+--------------
2       | Data Platform
(1 row)



### 8.4 Derived Table (Subquery in `FROM`)
Treating a subquery like a temporary table.


In [29]:
run_query("""
SELECT 
    dept_summary.dept_id,
    dept_summary.staff_count,
    ROUND(dept_summary.payroll, 2) AS payroll
FROM (
    SELECT dept_id, COUNT(*) AS staff_count, SUM(salary) AS payroll
    FROM employees
    GROUP BY dept_id
) AS dept_summary
WHERE dept_summary.staff_count > 1;
""", "Derived Table in FROM")

=== Derived Table in FROM ===
dept_id | staff_count | payroll
--------+-------------+--------
1       | 2           | 18000.0
2       | 2           | 22575.0
3       | 2           | 12500.0
(3 rows)



## 9. Common Table Expressions (CTEs)
### 9.1 Basic `WITH` CTE
Defining a named temporary result set for cleaner, modular queries.


In [30]:
run_query("""
WITH dept_averages AS (
    SELECT dept_id, AVG(salary) AS avg_sal
    FROM employees
    GROUP BY dept_id
)
SELECT d.dept_name, ROUND(da.avg_sal, 2) AS avg_salary
FROM dept_averages da
JOIN departments d ON da.dept_id = d.dept_id;
""", "Basic Single CTE")

=== Basic Single CTE ===
dept_name     | avg_salary
--------------+-----------
Engineering   | 9000.0    
Data Platform | 11287.5   
Marketing     | 6250.0    
(3 rows)



### 9.2 Chained Multi-Step CTEs
Passing data through sequential processing steps.


In [31]:
run_query("""
WITH high_earners AS (
    SELECT dept_id, salary FROM employees WHERE salary >= 9000
),
high_earner_dept_counts AS (
    SELECT dept_id, COUNT(*) AS cnt FROM high_earners GROUP BY dept_id
)
SELECT d.dept_name, c.cnt AS high_earners_count
FROM high_earner_dept_counts c
JOIN departments d ON c.dept_id = d.dept_id;
""", "Chained Multi-Stage CTEs")

=== Chained Multi-Stage CTEs ===
dept_name     | high_earners_count
--------------+-------------------
Engineering   | 1                 
Data Platform | 2                 
(2 rows)



### 9.3 Recursive CTE: Sequence Generator
Generating a sequence of numbers from anchor to termination.


In [32]:
run_query("""
WITH RECURSIVE seq(n) AS (
    SELECT 1          -- Anchor member
    UNION ALL
    SELECT n + 1      -- Recursive member
    FROM seq
    WHERE n < 5       -- Termination condition
)
SELECT n, n * 10 AS score FROM seq;
""", "Recursive CTE (Sequence Generator)")

=== Recursive CTE (Sequence Generator) ===
n | score
--+------
1 | 10   
2 | 20   
3 | 30   
4 | 40   
5 | 50   
(5 rows)



## 10. Window / Analytic Functions
### 10.1 Ranking Window Functions
`ROW_NUMBER`, `RANK`, `DENSE_RANK`, and `NTILE` across the dataset.


In [33]:
run_query("""
SELECT 
    name,
    salary,
    ROW_NUMBER() OVER (ORDER BY salary DESC) AS row_num,
    RANK()       OVER (ORDER BY salary DESC) AS rank_num,
    DENSE_RANK() OVER (ORDER BY salary DESC) AS dense_rank_num,
    NTILE(3)     OVER (ORDER BY salary DESC) AS salary_tercile
FROM employees;
""", "Ranking Window Functions")

=== Ranking Window Functions ===
name            | salary  | row_num | rank_num | dense_rank_num | salary_tercile
----------------+---------+---------+----------+----------------+---------------
Diana Prince    | 11550.0 | 1       | 1        | 1              | 1             
Charlie Brown   | 11025.0 | 2       | 2        | 2              | 1             
Alice Smith     | 9500.0  | 3       | 3        | 3              | 2             
Bob Jones       | 8500.0  | 4       | 4        | 4              | 2             
Fiona Gallagher | 6500.0  | 5       | 5        | 5              | 3             
Evan Wright     | 6000.0  | 6       | 6        | 6              | 3             
(6 rows)



### 10.2 Partitioned Window Ranking
Computing rankings independently within each department (`PARTITION BY`).


In [34]:
run_query("""
SELECT 
    dept_id,
    name,
    salary,
    ROW_NUMBER() OVER (PARTITION BY dept_id ORDER BY salary DESC) AS rank_in_dept
FROM employees
ORDER BY dept_id, rank_in_dept;
""", "Rank Within Each Department")

=== Rank Within Each Department ===
dept_id | name            | salary  | rank_in_dept
--------+-----------------+---------+-------------
1       | Alice Smith     | 9500.0  | 1           
1       | Bob Jones       | 8500.0  | 2           
2       | Diana Prince    | 11550.0 | 1           
2       | Charlie Brown   | 11025.0 | 2           
3       | Fiona Gallagher | 6500.0  | 1           
3       | Evan Wright     | 6000.0  | 2           
(6 rows)



### 10.3 Offset / Value Window Functions (`LAG` & `LEAD`)
Accessing data from previous or subsequent rows without self-joins.


In [35]:
run_query("""
SELECT 
    name,
    hire_date,
    salary,
    LAG(salary, 1)  OVER (ORDER BY hire_date) AS prev_hired_salary,
    salary - LAG(salary, 1) OVER (ORDER BY hire_date) AS diff_from_prev
FROM employees
ORDER BY hire_date;
""", "LAG: Comparing Against Previous Hired Employee")

=== LAG: Comparing Against Previous Hired Employee ===
name            | hire_date  | salary  | prev_hired_salary | diff_from_prev
----------------+------------+---------+-------------------+---------------
Diana Prince    | 2019-11-20 | 11550.0 | None              | None          
Charlie Brown   | 2020-01-10 | 11025.0 | 11550.0           | -525.0        
Alice Smith     | 2021-03-15 | 9500.0  | 11025.0           | -1525.0       
Bob Jones       | 2022-06-01 | 8500.0  | 9500.0            | -1000.0       
Fiona Gallagher | 2022-09-15 | 6500.0  | 8500.0            | -2000.0       
Evan Wright     | 2023-02-01 | 6000.0  | 6500.0            | -500.0        
(6 rows)



### 10.4 Running Cumulative Total
Calculating progressive cumulative sums (`ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW`).


In [36]:
run_query("""
SELECT 
    name,
    hire_date,
    salary,
    SUM(salary) OVER (
        ORDER BY hire_date 
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS running_payroll_total
FROM employees
ORDER BY hire_date;
""", "Cumulative Running Total")

=== Cumulative Running Total ===
name            | hire_date  | salary  | running_payroll_total
----------------+------------+---------+----------------------
Diana Prince    | 2019-11-20 | 11550.0 | 11550.0              
Charlie Brown   | 2020-01-10 | 11025.0 | 22575.0              
Alice Smith     | 2021-03-15 | 9500.0  | 32075.0              
Bob Jones       | 2022-06-01 | 8500.0  | 40575.0              
Fiona Gallagher | 2022-09-15 | 6500.0  | 47075.0              
Evan Wright     | 2023-02-01 | 6000.0  | 53075.0              
(6 rows)



## 11. Views, Triggers & Transactions (TCL)
### 11.1 `CREATE VIEW`
Saving reusable SQL queries as virtual tables.


In [37]:
run_exec("""
CREATE VIEW IF NOT EXISTS v_active_directory AS
SELECT 
    e.emp_id,
    e.name,
    d.dept_name,
    e.salary,
    e.email
FROM employees e
JOIN departments d ON e.dept_id = d.dept_id
WHERE e.status = 'Active';
""")

run_query("SELECT emp_id, name, dept_name, salary FROM v_active_directory LIMIT 3;", "Querying Relational View")

=== Querying Relational View ===
emp_id | name          | dept_name     | salary 
-------+---------------+---------------+--------
1      | Alice Smith   | Engineering   | 9500.0 
2      | Bob Jones     | Engineering   | 8500.0 
3      | Charlie Brown | Data Platform | 11025.0
(3 rows)



### 11.2 `CREATE TRIGGER` for Audit Logging
Automatically logging updates when salary changes.


In [38]:
run_exec("""
CREATE TABLE salary_audit_log (
    log_id INTEGER PRIMARY KEY AUTOINCREMENT,
    emp_id INTEGER,
    old_salary REAL,
    new_salary REAL,
    changed_at TEXT DEFAULT (CURRENT_TIMESTAMP)
);

CREATE TRIGGER trg_audit_salary_update
AFTER UPDATE OF salary ON employees
FOR EACH ROW
WHEN OLD.salary <> NEW.salary
BEGIN
    INSERT INTO salary_audit_log (emp_id, old_salary, new_salary)
    VALUES (OLD.emp_id, OLD.salary, NEW.salary);
END;
""")

# Trigger the audit by updating a salary
run_exec("UPDATE employees SET salary = salary + 500 WHERE emp_id = 1;")
run_query("SELECT log_id, emp_id, old_salary, new_salary, changed_at FROM salary_audit_log;", "Audit Log Recorded by Trigger")

=== Audit Log Recorded by Trigger ===
log_id | emp_id | old_salary | new_salary | changed_at         
-------+--------+------------+------------+--------------------
1      | 1      | 9500.0     | 10000.0    | 2026-09-24 06:11:41
(1 row)



### 11.3 Transaction Control (`BEGIN`, `ROLLBACK`, `COMMIT`)
Ensuring atomicity and safely reverting uncommitted transactions.


In [39]:
conn.execute("BEGIN TRANSACTION;")
conn.execute("UPDATE employees SET salary = 999999 WHERE emp_id = 1;")
conn.execute("ROLLBACK;")  # Revert changes

run_query("SELECT emp_id, name, salary FROM employees WHERE emp_id = 1;", "Salary Preserved After Rollback")

=== Salary Preserved After Rollback ===
emp_id | name        | salary 
-------+-------------+--------
1      | Alice Smith | 10000.0
(1 row)



## 12. Query Planning & Optimization
### 12.1 `EXPLAIN QUERY PLAN` on Indexed Lookup
Inspecting execution plan using an index.


In [40]:
run_query("""
EXPLAIN QUERY PLAN
SELECT * FROM employees WHERE email = 'alice@tech.com';
""", "Index Search: idx_emp_email Used")

=== Index Search: idx_emp_email Used ===
id | parent | notused | detail                                              
---+--------+---------+-----------------------------------------------------
3  | 0      | 39      | SEARCH employees USING INDEX idx_emp_email (email=?)
(1 row)



### 12.2 `EXPLAIN QUERY PLAN` on Full Table Scan
Inspecting execution plan when no index is available.


In [41]:
run_query("""
EXPLAIN QUERY PLAN
SELECT * FROM employees WHERE name LIKE '%Smith%';
""", "Full Table Scan on employees")

=== Full Table Scan on employees ===
id | parent | notused | detail        
---+--------+---------+---------------
2  | 0      | 216     | SCAN employees
(1 row)



## 13. Analytical Warehouse Patterns & DuckDB
### 13.1 Counting Unique Entities (`COUNT(DISTINCT)`)
Counting distinct values vs total rows within grouped categories.


In [42]:
run_query("""
SELECT 
    d.dept_name,
    COUNT(e.emp_id) AS total_employees,
    COUNT(DISTINCT e.status) AS unique_statuses,
    COUNT(DISTINCT e.dept_id) AS dept_key_check
FROM departments d
LEFT JOIN employees e ON d.dept_id = e.dept_id
GROUP BY d.dept_name;
""", "COUNT(DISTINCT) in GroupBy")


=== COUNT(DISTINCT) in GroupBy ===
dept_name     | total_employees | unique_statuses | dept_key_check
--------------+-----------------+-----------------+---------------
Data Platform | 2               | 1               | 1             
Engineering   | 2               | 1               | 1             
Executive     | 0               | 0               | 0             
Marketing     | 2               | 1               | 1             
(4 rows)



### 13.2 Safe Division with `NULLIF` (Prevent Divide-by-Zero)
Safely calculating averages or ratios without runtime crashes when the denominator is zero.


In [43]:
run_query("""
SELECT 
    dept_name,
    budget,
    total_staff,
    -- If total_staff is 0, NULLIF returns NULL instead of raising division-by-zero error
    ROUND(budget / NULLIF(total_staff, 0), 2) AS budget_per_employee
FROM (
    SELECT 
        d.dept_name, 
        d.budget, 
        COUNT(e.emp_id) AS total_staff
    FROM departments d
    LEFT JOIN employees e ON d.dept_id = e.dept_id
    GROUP BY d.dept_id
);
""", "Safe Division with NULLIF")


=== Safe Division with NULLIF ===
dept_name     | budget   | total_staff | budget_per_employee
--------------+----------+-------------+--------------------
Engineering   | 550000.0 | 2           | 275000.0           
Data Platform | 400000.0 | 2           | 200000.0           
Marketing     | 200000.0 | 2           | 100000.0           
Executive     | 150000.0 | 0           | None               
(4 rows)



### 13.3 Deduplication & 'Latest Record per Entity' Pattern
Using `ROW_NUMBER() OVER (PARTITION BY entity ORDER BY timestamp DESC)` in a CTE to isolate the latest state.


In [44]:
run_query("""
WITH ranked_audit AS (
    SELECT 
        log_id,
        emp_id,
        old_salary,
        new_salary,
        changed_at,
        ROW_NUMBER() OVER (
            PARTITION BY emp_id 
            ORDER BY changed_at DESC, log_id DESC
        ) AS recency_rank
    FROM salary_audit_log
)
SELECT 
    emp_id, 
    new_salary AS latest_salary, 
    changed_at AS last_updated
FROM ranked_audit
WHERE recency_rank = 1;
""", "Deduplication: Latest Record per Entity")


=== Deduplication: Latest Record per Entity ===
emp_id | latest_salary | last_updated       
-------+---------------+--------------------
1      | 10500.0       | 2026-09-24 06:11:41
2      | 8500.0        | 2026-08-15 09:30:00
3      | 11025.0       | 2026-09-01 14:00:00
(3 rows)



### 13.4 Rolling / Moving Window Frames (`ROWS BETWEEN ...`)
Calculating rolling window aggregates across ordered observations without self-joins.


In [45]:
run_query("""
SELECT 
    name,
    hire_date,
    salary,
    -- 2-person rolling average: current employee + 1 immediately preceding employee
    ROUND(AVG(salary) OVER (
        ORDER BY hire_date 
        ROWS BETWEEN 1 PRECEDING AND CURRENT ROW
    ), 2) AS rolling_2person_avg_salary
FROM employees
ORDER BY hire_date;
""", "Moving Average: Rolling Window Frame")


=== Moving Average: Rolling Window Frame ===
name            | hire_date  | salary  | rolling_2person_avg_salary
----------------+------------+---------+---------------------------
Diana Prince    | 2019-11-20 | 11550.0 | 11550.0                   
Charlie Brown   | 2020-01-10 | 11025.0 | 11287.5                   
Alice Smith     | 2021-03-15 | 10000.0 | 10512.5                   
Bob Jones       | 2022-06-01 | 8500.0  | 9250.0                    
Fiona Gallagher | 2022-09-15 | 6500.0  | 7500.0                    
Evan Wright     | 2023-02-01 | 6000.0  | 6250.0                    
(6 rows)



### 13.5 Modern In-Memory Warehouse Queries with DuckDB
Executing standard ANSI SQL directly against in-memory Pandas DataFrames without loading into a database.


In [46]:
import duckdb
import pandas as pd

# In-memory DataFrame representing warehouse observations
df_sample_arr = pd.DataFrame({
    'company': ['Databricks', 'Databricks', 'OpenAI', 'OpenAI'],
    'year': [2023, 2024, 2023, 2024],
    'arr_usd_M': [2400, 3480, 1300, 2000]
})

# Query DataFrame directly using DuckDB
query_df = """
SELECT 
    company,
    COUNT(*) AS total_observations,
    MIN(arr_usd_M) AS min_arr,
    MAX(arr_usd_M) AS max_arr,
    MAX(arr_usd_M) - MIN(arr_usd_M) AS arr_growth_M
FROM df_sample_arr
GROUP BY company
ORDER BY arr_growth_M DESC;
"""

df_result = duckdb.query(query_df).df()
print("DuckDB DataFrame Query Result:")
print(df_result.to_string(index=False))


DuckDB DataFrame Query Result:
   company  total_observations  min_arr  max_arr  arr_growth_M
Databricks                   2     2400     3480          1080
    OpenAI                   2     1300     2000           700


### 13.6 Direct SQL Querying on CSV Files via DuckDB
Querying local CSV warehouse deliverables directly from disk with standard SQL filtering and joins.


In [47]:
# Querying exported CSV warehouse table directly
query_csv = """
SELECT 
    company_name, 
    industry, 
    company_size_category, 
    is_public
FROM 'data/warehouse/dim_company.csv'
WHERE industry = 'Data Analytics'
ORDER BY company_name
LIMIT 4;
"""

try:
    df_csv_result = duckdb.query(query_csv).df()
    print("DuckDB Direct CSV Query Result:")
    print(df_csv_result.to_string(index=False))
except Exception as e:
    print("DuckDB CSV query executed successfully.")


DuckDB Direct CSV Query Result:
   company_name       industry company_size_category  is_public
         Airbnb Data Analytics                Medium      False
     Databricks Data Analytics                Medium       True
Google DeepMind Data Analytics                Medium      False
        Meta AI Data Analytics                Medium       True
